In [2]:
# libraries
import mysql.connector
import json
import chromadb
import sqlparse
from chromadb.utils.embedding_functions import SentenceTransformerEmbeddingFunction
from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate

In [64]:
# CONSTANTS
LLM_API_KEY = ''
DATABASE_NAME = "ecommerce_db"
COLLECTION_NAME = "New_Table_Schema_of_Ecommerce_Database"
EMBEDDING_MODEL_NAME = 'all-MiniLM-L6-v2'
LLM_MODEL = 'llama-3.3-70b-versatile'

In [43]:
# Connect to MySQL
conn = mysql.connector.connect(
    host="localhost",
    user="root",
    password="",
    database=f"{DATABASE_NAME}"  # optional
)
cursor = conn.cursor()

# Get the tables in the database
cursor.execute(f"USE {DATABASE_NAME}")
cursor.execute("SHOW TABLES")
tables = [row[0] for row in cursor.fetchall()]

# Get complete schema
def extract_table_info(tables: list):
    metadata = []
    table_info = []
    
    for table in tables:
        cursor.execute(f"DESCRIBE {table}")
        for col in cursor.fetchall():
            chunk = f"""
Database: ecommerce_db
Table: {table}
Column: {col[0]}
Type: {col[1]}
Null Constraint: {col[2]}
Key Type: {col[3]}
Default Value: {col[4]}
Extra Constraints: {col[5]}
            """
            table_info.append(chunk)
            
        # metadata.append
    return table_info

info = extract_table_info(tables)

In [50]:
for i in range(len(info)):
    if info[i].split()[3] == "products":
        print(info[i])


Database: ecommerce_db
Table: products
Column: product_id
Type: int unsigned
Null Constraint: NO
Key Type: PRI
Default Value: None
Extra Constraints: auto_increment
            

Database: ecommerce_db
Table: products
Column: category_id
Type: int unsigned
Null Constraint: NO
Key Type: MUL
Default Value: None
Extra Constraints: 
            

Database: ecommerce_db
Table: products
Column: sku
Type: varchar(60)
Null Constraint: NO
Key Type: UNI
Default Value: None
Extra Constraints: 
            

Database: ecommerce_db
Table: products
Column: name
Type: varchar(200)
Null Constraint: NO
Key Type: 
Default Value: None
Extra Constraints: 
            

Database: ecommerce_db
Table: products
Column: description
Type: text
Null Constraint: YES
Key Type: 
Default Value: None
Extra Constraints: 
            

Database: ecommerce_db
Table: products
Column: brand
Type: varchar(80)
Null Constraint: YES
Key Type: MUL
Default Value: None
Extra Constraints: 
            

Database: ecommerce_db
Ta

In [45]:
print(len(info))

106


In [71]:
# Vector Database creation
client = chromadb.PersistentClient()
# collection creation
try:
    client.delete_collection(name = collection_name)
except:
    pass
    
collection = client.get_or_create_collection(
    name = COLLECTION_NAME,
    embedding_function= SentenceTransformerEmbeddingFunction(model_name=EMBEDDING_MODEL_NAME)
)

# add records to the collection
ids = [f"id{i+1}" for i in range(len(info))]
collection.add(
    ids=ids,
    documents=info
)

# user query
user_query = "Qeury to list all the persons with max sales year over year"

# Query Vector Database
query = collection.query(
    query_texts=user_query,
    n_results=8
)

# Combine all the chunks
document_text = '\n\n'.join(query['documents'][0])

In [72]:
for i in query['documents'][0]:
    print(i)


Database: ecommerce_db
Table: coupons
Column: max_uses
Type: int
Null Constraint: YES
Key Type: 
Default Value: None
Extra Constraints: 
            

Database: ecommerce_db
Table: coupons
Column: expires_at
Type: datetime
Null Constraint: YES
Key Type: MUL
Default Value: None
Extra Constraints: 
            

Database: ecommerce_db
Table: coupons
Column: per_user_limit
Type: tinyint
Null Constraint: NO
Key Type: 
Default Value: 1
Extra Constraints: 
            

Database: ecommerce_db
Table: customers
Column: date_of_birth
Type: date
Null Constraint: YES
Key Type: 
Default Value: None
Extra Constraints: 
            

Database: ecommerce_db
Table: coupons
Column: starts_at
Type: datetime
Null Constraint: NO
Key Type: 
Default Value: None
Extra Constraints: 
            

Database: ecommerce_db
Table: coupons
Column: used_count
Type: int
Null Constraint: NO
Key Type: 
Default Value: 0
Extra Constraints: 
            

Database: ecommerce_db
Table: coupons
Column: created_at
Type: dat

In [73]:
# LLM model creation
# llama model
llm  = ChatGroq(
    temperature=0,
    api_key= LLM_API_KEY,
    model=LLM_MODEL,
    max_tokens= 3000
)

# prompt
prompt = PromptTemplate.from_template(
    """
    You are an expert MySQL SQL Assistant.

    Your task is to generate optimized, production-ready SQL queries.
    
    Rules:
    - Only generate SELECT queries
    - Never generate DELETE, UPDATE, INSERT, DROP, ALTER, or TRUNCATE queries
    - Use explicit column names instead of SELECT *
    - Use only the tables and columns provided in the schema
    - Do not hallucinate tables or columns
    - Generate syntactically correct MySQL queries
    - Prefer optimized joins and filtering
    - If the query cannot be generated from the schema, explain why
    
    Return the output STRICTLY in this format ONLL: 
    <SQL QUERY HERE>",
    
    DO NOT:
    - Return markdown
    - Return backticks
    - Return extra text
    - Return comments outside JSON
    
    User Question:
    {user_question}

    Content:
    {extracted_content}
    """
)

# chain creation
chain = prompt | llm

# response generation and display
response = chain.invoke(
    {
        "extracted_content": document_text,
        "user_question": user_query
    }
)

# formatting the sql 
formatted_sql = sqlparse.format(
    response.content,
    reindent=True,
    keyword_case="upper"
)

print(formatted_sql)

SELECT customers.date_of_birth,
       orders.created_at
FROM customers
INNER JOIN orders ON customers.id = orders.customer_id
ORDER BY YEAR(orders.created_at) DESC
LIMIT 1
